# PBS Emulator Training

This notebook trains a reusable physics-based simulation (PBS) emulator from generated degradation trajectories and exports the figure panels used to summarize dataset coverage and emulator accuracy.


## Module 1. Configuration


In [ ]:
from pathlib import Path

PBS_DATASET_ID = "pbs_lhs_400"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_ROOT / "figures"
CACHE_ROOT = OUTPUT_ROOT / "cache"
PBS_DATA_ROOT = DATA_ROOT / "pbs" / PBS_DATASET_ID
DATASET_CACHE_DIR = CACHE_ROOT / PBS_DATASET_ID

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SCAN_SUMMARY_CSV = PBS_DATA_ROOT / "scan_summary.csv"
SCAN_METADATA_JSON = PBS_DATA_ROOT / "scan_metadata.json"
LIFE_SOH_DIR = PBS_DATA_ROOT / "life_soh_csv"
CYCLE_QV_DIR = PBS_DATA_ROOT / "cycle_qv_csv"

PARAMETER_REPRESENTATION = "multiplier"  # "multiplier" or "value"
MODEL_TAG = f"pbs_emulator_{PBS_DATASET_ID}_{PARAMETER_REPRESENTATION}_linearcycle_v1"
ARTIFACT_PATH = DATASET_CACHE_DIR / f"{MODEL_TAG}.joblib"
METRICS_CSV_PATH = DATASET_CACHE_DIR / f"{MODEL_TAG}_metrics.csv"
LEARNING_CURVE_CSV_PATH = DATASET_CACHE_DIR / f"{MODEL_TAG}_learning_curve.csv"
EXTRAPOLATION_CSV_PATH = DATASET_CACHE_DIR / f"{MODEL_TAG}_edge_extrapolation.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print(f"Project root: {PROJECT_ROOT}")
print(f"PBS data root: {PBS_DATA_ROOT}")
print(f"Emulator artifact path: {ARTIFACT_PATH}")


## Module 2. Imports and helpers

These helpers are specialized for the generated PBS dataset layout. The workflow is:
- read parameter definitions from `scan_metadata.json`
- read sampled parameter values from `scan_summary.csv`
- reconstruct SOH-threshold crossing cycles from each `life_soh_csv/sim_xxxxxx.csv`
- convert SOH cycles to log space, then to monotonic increments for NN training


In [ ]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler

class PbsDegradationDataset:
    def __init__(self, root, parameter_representation="multiplier"):
        self.root = Path(root)
        self.summary_csv = self.root / "scan_summary.csv"
        self.metadata_json = self.root / "scan_metadata.json"
        self.life_dir = self.root / "life_soh_csv"
        self.parameter_representation = str(parameter_representation)

        if not self.summary_csv.exists():
            raise FileNotFoundError(self.summary_csv)
        if not self.metadata_json.exists():
            raise FileNotFoundError(self.metadata_json)
        if not self.life_dir.exists():
            raise FileNotFoundError(self.life_dir)

        self.metadata = json.loads(self.metadata_json.read_text(encoding="utf-8"))
        self.summary_df = pd.read_csv(self.summary_csv)
        self.param_names = list(self.metadata["pm_variation_list"])
        self.soh_thresholds = [int(x) for x in self.metadata["soh_thresholds"]]

        if self.parameter_representation not in {"multiplier", "value"}:
            raise ValueError("parameter_representation must be `multiplier` or `value`.")

    def get_param_columns(self):
        suffix = "__multiplier" if self.parameter_representation == "multiplier" else "__value"
        cols = [f"{name}{suffix}" for name in self.param_names]
        missing = [col for col in cols if col not in self.summary_df.columns]
        if missing:
            raise ValueError(f"Missing parameter columns in scan_summary.csv: {missing}")
        return cols

    def resolve_relative_csv(self, rel_path):
        rel_parts = str(rel_path).replace(chr(92), "/").split("/")
        return self.root.joinpath(*[part for part in rel_parts if part])

    def load_life_curve(self, simulation_index=None, rel_path=None):
        if rel_path is None:
            rel_path = f"life_soh_csv/sim_{int(simulation_index):06d}.csv"
        path = self.resolve_relative_csv(rel_path)
        return pd.read_csv(path)

    def extract_soh_crossing_cycles(self, life_df, thresholds=None):
        if thresholds is None:
            thresholds = self.soh_thresholds
        soh = pd.to_numeric(life_df["soh"], errors="coerce").to_numpy(dtype=float)
        cycle = pd.to_numeric(life_df["cycle_number"], errors="coerce").to_numpy(dtype=float)
        out = []
        for level in thresholds:
            mask = np.isfinite(soh) & np.isfinite(cycle) & (soh <= float(level))
            if np.any(mask):
                out.append(float(cycle[np.flatnonzero(mask)[0]]))
            else:
                out.append(float("nan"))
        return np.asarray(out, dtype=float)

    def build_training_table(self):
        param_cols = self.get_param_columns()
        rows = []
        for row in self.summary_df.to_dict(orient="records"):
            sim_idx = int(row["simulation_index"])
            life_df = self.load_life_curve(rel_path=row["life_detail_csv"])
            soh_cycles = self.extract_soh_crossing_cycles(life_df)

            record = {
                "simulation_index": sim_idx,
                "80soh_cycle": float(row.get("80soh_cycle", np.nan)),
            }
            for col in param_cols:
                record[col] = float(row[col])
            for threshold, cyc in zip(self.soh_thresholds, soh_cycles):
                record[f"{int(threshold)}% SOH"] = float(cyc) if np.isfinite(cyc) and cyc > 0 else float("nan")
            rows.append(record)
        return pd.DataFrame(rows)


def build_monotonic_forward_targets(y_raw, soh_count):
    soh_cycles = y_raw[:, :soh_count]
    base_cycle = soh_cycles[:, [0]]
    delta_cycles = np.diff(soh_cycles, axis=1)
    delta_cycles = np.clip(delta_cycles, 1e-12, None)
    return np.hstack([base_cycle, delta_cycles])


def invert_monotonic_forward_targets(y_transformed, soh_count):
    base_cycle = y_transformed[:, [0]]
    delta_cycles = np.clip(y_transformed[:, 1:soh_count], 1e-12, None)
    soh_cycles = np.concatenate([base_cycle, base_cycle + np.cumsum(delta_cycles, axis=1)], axis=1)
    return np.clip(soh_cycles, 1e-12, None)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def monotonic_ok_mask_from_raw_predictions(y_pred_raw, soh_count):
    soh_cycles = y_pred_raw[:, :soh_count]
    return np.all(np.diff(soh_cycles, axis=1) >= -1e-10, axis=1)


def load_artifact(path):
    if not Path(path).exists():
        raise FileNotFoundError(path)
    return joblib.load(path)


In [ ]:
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"]
})


## Module 3. Load and parse the generated PBS dataset

This module builds the canonical SOH-only training table directly from the generated simulation folders, so the surrogate no longer depends on a manually assembled CSV.


In [ ]:
dataset = PbsDegradationDataset(PBS_DATA_ROOT, parameter_representation=PARAMETER_REPRESENTATION)
pbs_data = dataset.build_training_table()
param_cols = dataset.get_param_columns()
param_names = list(dataset.param_names)
param_col_to_name = dict(zip(param_cols, param_names))
soh_cols = [f"{int(level)}% SOH" for level in dataset.soh_thresholds]
extra_target_cols = []
nn_target_cols = soh_cols.copy()

working_df = pbs_data.copy()
working_df = working_df.rename(columns=param_col_to_name)

for col in param_names + nn_target_cols:
    working_df[col] = pd.to_numeric(working_df[col], errors="coerce")

model_df = working_df[param_names + ["simulation_index", "80soh_cycle"] + nn_target_cols].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
X_param = model_df[param_names].to_numpy(dtype=float)
Y_target_raw = model_df[nn_target_cols].to_numpy(dtype=float)

print(f"Loaded {len(pbs_data)} generated simulations from {PBS_DATA_ROOT}")
print(f"Usable training rows after cleanup: {len(model_df)}")
print(f"Parameter columns ({PARAMETER_REPRESENTATION}): {param_names}")
print(f"SOH target columns: {nn_target_cols[:5]} ...")
display(model_df.head())


## Module 4. Quick audit of the PyBaMM parameter and SOH-target space


In [ ]:
display(model_df[param_names + ["80soh_cycle"]].describe().T)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
axes = axes.ravel()
for ax, name in zip(axes, param_names):
    sns.histplot(model_df[name], bins=30, ax=ax, kde=True, color="#356a9a")
    ax.set_title(name)
for ax in axes[len(param_names):]:
    ax.set_visible(False)
plt.savefig(FIGURE_DIR / "S1.tiff", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ---------- Data processing (unchanged) ----------
full_surrogate_soh_cols = sorted(
    [col for col in nn_target_cols if str(col).endswith("% SOH")],
    key=lambda col: int(str(col).split("%")[0]),
    reverse=True,
)
full_surrogate_soh_levels = [int(str(col).split("%")[0]) for col in full_surrogate_soh_cols]

trajectory_rows = []
for _, row in model_df[["simulation_index"] + full_surrogate_soh_cols].iterrows():
    sim_idx = int(row["simulation_index"])
    cycles = row[full_surrogate_soh_cols].to_numpy(dtype=float)
    valid = np.isfinite(cycles) & (cycles > 0)
    if not np.any(valid):
        continue

    levels = np.asarray(full_surrogate_soh_levels, dtype=float)[valid]
    cycles = cycles[valid]

    plot_cycles = np.concatenate([[0.0], cycles])
    plot_soh = np.concatenate([[100.0], levels])
    cycle_at_80 = float(cycles[levels == 80][0]) if np.any(levels == 80) else float(cycles[-1])

    for cyc, soh in zip(plot_cycles, plot_soh):
        trajectory_rows.append({
            "simulation_index": sim_idx,
            "cycle": float(cyc),
            "soh": float(soh),
            "cycle_at_80": cycle_at_80,
        })

pybamm_trajectory_df = pd.DataFrame(trajectory_rows)
if pybamm_trajectory_df.empty:
    raise ValueError("No valid PyBaMM SOH trajectories were found for plotting.")

surrogate_dataset_max_cycle = float(pybamm_trajectory_df["cycle_at_80"].max())
if surrogate_dataset_max_cycle <= 0:
    raise ValueError("Surrogate dataset max cycle must be positive.")

pybamm_trajectory_df["norm_cycle"] = pybamm_trajectory_df["cycle"] / surrogate_dataset_max_cycle

# ---------- Prepare data for the upper histogram ----------
# Extract cycle_at_80 values and normalise them by the max
cycle_80_values = pybamm_trajectory_df.groupby("simulation_index")["cycle_at_80"].first().values
norm_cycle_80_values = cycle_80_values / surrogate_dataset_max_cycle   # 0 to 1

# ---------- Plotting: main + upper + colorbar ----------
cmap = plt.cm.coolwarm
norm = Normalize(
    pybamm_trajectory_df["cycle_at_80"].min(),
    pybamm_trajectory_df["cycle_at_80"].max(),
)

fig = plt.figure(figsize=(5, 4.8), dpi=500)
gs = fig.add_gridspec(2, 2, width_ratios=[4, 0.25], height_ratios=[1,5],
                      hspace=0.05, wspace=0.05)   # tight vertical spacing
ax_main = fig.add_subplot(gs[1, 0])
ax_upper = fig.add_subplot(gs[0, 0])

# ---------- 1. Main plot: trajectories ----------
for sim_idx, grp in pybamm_trajectory_df.groupby("simulation_index"):
    grp = grp.sort_values("soh", ascending=False)
    cycle_at_80 = float(grp["cycle_at_80"].iloc[0])
    color = cmap(norm(cycle_at_80))
    ax_main.plot(
        grp["norm_cycle"],
        grp["soh"],
        color=color,
        alpha=0.6,
        linewidth=1,
        marker="o",
        markersize=3,
        markeredgewidth=0,
    )

ax_main.set_xlabel("")                     # Remove x‑label from main plot
ax_main.set_ylabel("State of health (SOH%)",fontsize=12)
ax_main.set_xlabel("Normalized cycle", fontsize=12)                     
ax_main.set_yticks([80, 85, 90, 95, 100])
ax_main.set_ylim(79, 101)

# ---------- 2. upper histogram: x‑axis normalised (0‑1), y‑axis = count, no y‑label ----------
counts, bin_edges = np.histogram(norm_cycle_80_values, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]

# Colour each bar according to its original (unnormalised) cycle_at_80 value.
original_bin_centers = bin_centers * surrogate_dataset_max_cycle
bar_colors = cmap(norm(original_bin_centers))

# Bars touch each other (width = bin_width)
ax_upper.bar(bin_centers, counts, width=bin_width,
              color=bar_colors, edgecolor='None', linewidth=0.5)
ax_upper.set_ylabel("")
ax_upper.set_yticks([])  # No y‑ticks                                          
ax_upper.tick_params(length=0)
ax_upper.set_xticklabels([])               # Remove x‑tick labels from main plot
# Instead of ax_main.set_xlim(0, 1) and ax_upper.set_xlim(0, 1), use:
x_pad = 0.05
ax_main.set_xlim(-x_pad, 1 + x_pad)
ax_upper.set_xlim(-x_pad, 1 + x_pad)

# Keep the same tick positions (0, 0.2, 0.4, 0.6, 0.8, 1.0)
tick_locs = np.arange(0, 1.01, 0.2)
ax_main.set_xticks(tick_locs)
ax_upper.set_xticks(tick_locs)

ax_upper.text(0.98, 0.9, "80%SOH cycle distribution", 
               transform=ax_upper.transAxes, ha='right', va='top',
               fontsize=10)

plt.savefig(FIGURE_DIR / "F2a.tiff", dpi=500, format="tiff", bbox_inches="tight")
plt.show()

print(f"Plotted {pybamm_trajectory_df['simulation_index'].nunique()} PyBaMM trajectories")
print(f"Surrogate normalization basis: {surrogate_dataset_max_cycle:.2f} cycles")


## Module 5. Absolute parameter-to-SOH correlation trajectories


In [ ]:
corr_rows = []
soh_levels = [int(col.split('%')[0]) for col in soh_cols]

for param_name in param_names:
    for soh_level, soh_col in zip(soh_levels, soh_cols):
        valid = model_df[[param_name, soh_col]].dropna()
        if len(valid) >= 3 and valid[param_name].nunique() > 1 and valid[soh_col].nunique() > 1:
            corr_value = float(valid[param_name].corr(valid[soh_col]))
        else:
            corr_value = float("nan")
        corr_rows.append({
            "param": param_name,
            "soh_level": soh_level,
            "abs_corr": abs(corr_value) if np.isfinite(corr_value) else float("nan"),
        })

param_soh_corr_df = pd.DataFrame(corr_rows)
marker_cycle = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
param_markers = {param_name: marker_cycle[i % len(marker_cycle)] for i, param_name in enumerate(param_names)}

fig, ax = plt.subplots(figsize=(8.0, 5.0), constrained_layout=True)
for param_name in param_names:
    sub = param_soh_corr_df.loc[param_soh_corr_df["param"].eq(param_name)].sort_values("soh_level", ascending=False)
    ax.plot(
        sub["soh_level"],
        sub["abs_corr"],
        marker=param_markers[param_name],
        markersize=5.5,
        linewidth=1.6,
        label=param_name,
    )

ax.set_title("Absolute parameter-SOH correlation across SOH levels")
ax.set_xlabel("SOH level (%)")
ax.set_ylabel("|Pearson r|")
ax.set_ylim(0.0, 1.05)
ax.invert_xaxis()
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=8, loc="best")
plt.show()


## Module 5A. Absolute parameter-to-segment-slope correlation trajectories

This module uses the surrogate SOH crossing cycles in linear cycle space, normalizes them by the global maximum cycle across the surrogate dataset, and then computes four coarse segment slopes: `100->95`, `95->90`, `90->85`, and `85->80`. The plot shows the absolute Pearson correlation between each parameter and each normalized segment slope.


In [ ]:
required_levels = [95, 90, 85, 80]
missing_levels = [level for level in required_levels if f"{level}% SOH" not in soh_cols]
if missing_levels:
    raise ValueError(f"Missing SOH columns required for segment-slope analysis: {missing_levels}")

soh_cycle_df = pd.DataFrame({
    "simulation_index": model_df["simulation_index"].to_numpy(dtype=int),
})
for col in soh_cols:
    soh_cycle_df[col] = model_df[col].to_numpy(dtype=float)

surrogate_global_max_cycle = float(np.nanmax(soh_cycle_df[soh_cols].to_numpy(dtype=float)))
if not np.isfinite(surrogate_global_max_cycle) or surrogate_global_max_cycle <= 0:
    raise ValueError("Surrogate global max cycle must be positive for slope normalization.")

segment_points = {100: np.zeros(len(model_df), dtype=float)}
for level in required_levels:
    segment_points[level] = soh_cycle_df[f"{level}% SOH"].to_numpy(dtype=float) / surrogate_global_max_cycle

segment_defs = [(100, 95), (95, 90), (90, 85), (85, 80)]
segment_rows = []
for start_level, end_level in segment_defs:
    seg_label = f"{start_level}->{end_level}"
    seg_slope = (segment_points[end_level] - segment_points[start_level]) / float(start_level - end_level)
    for param_name in param_names:
        valid_mask = np.isfinite(seg_slope) & np.isfinite(model_df[param_name].to_numpy(dtype=float))
        valid = pd.DataFrame({
            "param": model_df.loc[valid_mask, param_name].to_numpy(dtype=float),
            "segment_slope": seg_slope[valid_mask],
        })
        if len(valid) >= 3 and valid["param"].nunique() > 1 and valid["segment_slope"].nunique() > 1:
            corr_value = float(valid["param"].corr(valid["segment_slope"]))
        else:
            corr_value = float("nan")
        segment_rows.append({
            "param": param_name,
            "segment": seg_label,
            "abs_corr": abs(corr_value) if np.isfinite(corr_value) else float("nan"),
        })

segment_corr_df = pd.DataFrame(segment_rows)
segment_order = [f"{start}->{end}" for start, end in segment_defs]

fig, ax = plt.subplots(figsize=(8.0, 5.0), constrained_layout=True)
for param_name in param_names:
    sub = segment_corr_df.loc[segment_corr_df["param"].eq(param_name)].copy()
    sub["segment"] = pd.Categorical(sub["segment"], categories=segment_order, ordered=True)
    sub = sub.sort_values("segment")
    ax.plot(
        sub["segment"],
        sub["abs_corr"],
        marker=param_markers[param_name],
        markersize=5.5,
        linewidth=1.6,
        label=param_name,
    )

ax.set_title("Absolute parameter-segment-slope correlation")
ax.set_xlabel("Normalized SOH segment")
ax.set_ylabel("|Pearson r|")
ax.set_ylim(0.0, 1.05)
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=8, loc="best")
plt.show()


## Module 5B. Scatter distribution of normalized segment slopes across all emulator cells

This module visualizes the four normalized segment slopes for every surrogate cell, using the same segment definition as Module 5A. It is meant to show the spread and overlap of the surrogate trajectory pacing across `100->95`, `95->90`, `90->85`, and `85->80`.


In [ ]:
required_levels = [95, 90, 85, 80]
missing_levels = [level for level in required_levels if f"{level}% SOH" not in soh_cols]
if missing_levels:
    raise ValueError(f"Missing SOH columns required for segment-slope scatter analysis: {missing_levels}")

soh_cycle_df = pd.DataFrame({
    "simulation_index": model_df["simulation_index"].to_numpy(dtype=int),
})
for col in soh_cols:
    soh_cycle_df[col] = model_df[col].to_numpy(dtype=float)

surrogate_global_max_cycle = float(np.nanmax(soh_cycle_df[soh_cols].to_numpy(dtype=float)))
if not np.isfinite(surrogate_global_max_cycle) or surrogate_global_max_cycle <= 0:
    raise ValueError("Surrogate global max cycle must be positive for slope normalization.")

segment_points = {100: np.zeros(len(model_df), dtype=float)}
for level in required_levels:
    segment_points[level] = soh_cycle_df[f"{level}% SOH"].to_numpy(dtype=float) / surrogate_global_max_cycle

segment_defs = [(100, 95), (95, 90), (90, 85), (85, 80)]
scatter_rows = []
for start_level, end_level in segment_defs:
    seg_label = f"{start_level}->{end_level}"
    seg_slope = (segment_points[end_level] - segment_points[start_level]) / float(start_level - end_level)
    for sim_idx, slope_val in zip(model_df["simulation_index"].to_numpy(dtype=int), seg_slope):
        scatter_rows.append({
            "simulation_index": int(sim_idx),
            "segment": seg_label,
            "normalized_slope": float(slope_val),
        })

segment_slope_scatter_df = pd.DataFrame(scatter_rows)
segment_order = [f"{start}->{end}" for start, end in segment_defs]
segment_to_x = {seg: idx for idx, seg in enumerate(segment_order)}
rng = np.random.default_rng(10)
segment_slope_scatter_df["x_jitter"] = segment_slope_scatter_df["segment"].map(segment_to_x).astype(float) + rng.uniform(-0.12, 0.12, size=len(segment_slope_scatter_df))

fig, ax = plt.subplots(figsize=(8.0, 5.0), constrained_layout=True)
ax.scatter(
    segment_slope_scatter_df["x_jitter"],
    segment_slope_scatter_df["normalized_slope"],
    s=20,
    alpha=0.55,
    color="#356a9a",
    edgecolors="none",
)

segment_summary = segment_slope_scatter_df.groupby("segment")["normalized_slope"].agg(["median", "min", "max"]).reindex(segment_order)
for seg, row in segment_summary.iterrows():
    x = segment_to_x[seg]
    ax.plot([x, x], [row["min"], row["max"]], color="#b65f2a", linewidth=1.2, alpha=0.8)
    ax.scatter([x], [row["median"]], color="#b65f2a", s=36, zorder=3)

ax.set_title("Distribution of normalized segment slopes across emulator cells")
ax.set_xlabel("SOH segment")
ax.set_ylabel("Normalized slope")
ax.set_xticks(range(len(segment_order)))
ax.set_xticklabels(segment_order)
ax.grid(alpha=0.25)
plt.show()


## Module 5C. Targeted mechanism-pair relation with life

This module focuses only on three mechanism pairs of interest:

1. `Lithium plating kinetic rate constant [m.s-1]` and `Dead lithium decay constant [s-1]`
2. `Negative electrode LAM constant proportional term [s-1]` and `Negative electrode critical stress [Pa]`
3. `Initial SEI thickness [m]` and `SEI solvent diffusivity [m2.s-1]`

For each pair, the goal is not to screen all interactions, but to directly inspect whether the two parameters appear to:

- move in a **consistent / mutually promoting** way with respect to life, or
- act more like **parallel / compensatory** directions.

The module therefore reports:

- the correlation of each individual parameter with life;
- the correlation between the two parameters themselves;
- and a pairwise scatter plot colored by life.

Interpretation:

- if both parameters correlate with life in the same direction and are positively associated with each other, that pair is more consistent with a mutually promoting trend;
- if both parameters correlate with life but the pair itself is weakly related, they are more parallel;
- if one increases while the other decreases for similar life outcomes, that suggests compensation.


In [ ]:
target_pairs = [
    (
        "Lithium plating kinetic rate constant [m.s-1]",
        "Dead lithium decay constant [s-1]",
    ),
    (
        "Negative electrode LAM constant proportional term [s-1]",
        "Negative electrode critical stress [Pa]",
    ),
    (
        "Initial SEI thickness [m]",
        "SEI solvent diffusivity [m2.s-1]",
    ),
]

life_col = "80soh_cycle"
if life_col not in model_df.columns:
    raise ValueError(f"Expected life column {life_col!r} in model_df.")

missing_params = sorted({p for pair in target_pairs for p in pair if p not in model_df.columns})
if missing_params:
    raise ValueError(f"Missing expected parameter columns: {missing_params}")

summary_rows = []

fig, axes = plt.subplots(1, len(target_pairs), figsize=(5.1 * len(target_pairs), 4.4), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

life_vals_full = model_df[life_col].to_numpy(dtype=float)
life_vmin = float(np.nanmin(life_vals_full))
life_vmax = float(np.nanmax(life_vals_full))

for ax, (param_x, param_y) in zip(axes, target_pairs):
    sub = model_df[[param_x, param_y, life_col]].dropna().copy()
    x = sub[param_x].to_numpy(dtype=float)
    y = sub[param_y].to_numpy(dtype=float)
    life_vals = sub[life_col].to_numpy(dtype=float)

    corr_x_life = float(np.corrcoef(x, life_vals)[0, 1]) if len(sub) >= 3 and np.nanstd(x) > 1e-12 and np.nanstd(life_vals) > 1e-12 else float("nan")
    corr_y_life = float(np.corrcoef(y, life_vals)[0, 1]) if len(sub) >= 3 and np.nanstd(y) > 1e-12 and np.nanstd(life_vals) > 1e-12 else float("nan")
    corr_xy = float(np.corrcoef(x, y)[0, 1]) if len(sub) >= 3 and np.nanstd(x) > 1e-12 and np.nanstd(y) > 1e-12 else float("nan")

    if np.isfinite(corr_x_life) and np.isfinite(corr_y_life):
        if np.sign(corr_x_life) == np.sign(corr_y_life):
            if np.isfinite(corr_xy) and corr_xy > 0.25:
                relation_label = "consistent / mutually promoting"
            else:
                relation_label = "parallel but same-direction"
        else:
            relation_label = "compensatory / opposing"
    else:
        relation_label = "undetermined"

    summary_rows.append({
        "param_x": param_x,
        "param_y": param_y,
        "corr_x_life": corr_x_life,
        "corr_y_life": corr_y_life,
        "corr_xy": corr_xy,
        "inferred_relation": relation_label,
        "n_samples": int(len(sub)),
    })

    sc = ax.scatter(
        x,
        y,
        c=life_vals,
        cmap="coolwarm",
        vmin=life_vmin,
        vmax=life_vmax,
        s=60,
        alpha=0.82,
        edgecolors="none",
    )

    ax.set_xlabel(param_x)
    ax.set_ylabel(param_y)
    ax.set_title(
        f"{relation_label}\n"
        f"r(x,life)={corr_x_life:.2f}, r(y,life)={corr_y_life:.2f}, r(x,y)={corr_xy:.2f}",
        fontsize=9,
    )
    ax.grid(alpha=0.22)

cbar = fig.colorbar(sc, ax=axes, shrink=0.92)
cbar.set_label(life_col)
plt.show()

targeted_pair_relation_df = pd.DataFrame(summary_rows)
display(targeted_pair_relation_df)


## Module 6. Prepare monotonic emulator targets and train/test split


In [ ]:
n_soh_targets = len(soh_cols)
Y_target_monotonic = build_monotonic_forward_targets(Y_target_raw, n_soh_targets)

X_train_raw, X_test_raw, Y_train_raw, Y_test_raw, Y_train_monotonic, Y_test_monotonic = train_test_split(
    X_param,
    Y_target_raw,
    Y_target_monotonic,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

nn_x_scaler = MinMaxScaler()
nn_y_scaler = StandardScaler()
X_train_nn = nn_x_scaler.fit_transform(X_train_raw)
X_test_nn = nn_x_scaler.transform(X_test_raw)
Y_train_nn = nn_y_scaler.fit_transform(Y_train_monotonic)

print(f"Train rows: {len(X_train_raw)} | Test rows: {len(X_test_raw)}")


## Module 7. Train or load the cached emulator artifact


In [ ]:
force_retrain = False

if ARTIFACT_PATH.exists() and not force_retrain:
    artifact = load_artifact(ARTIFACT_PATH)
    nn_forward_model = artifact["model"]
    nn_x_scaler = artifact["x_scaler"]
    nn_y_scaler = artifact["y_scaler"]
    artifact_meta = artifact.get("metadata", {})

    cached_param_names = list(artifact_meta.get("param_names", []))
    cached_soh_cols = list(artifact_meta.get("soh_cols", []))
    cached_target_cols = list(artifact_meta.get("target_cols", []))
    cached_target_encoding = artifact_meta.get("target_encoding")
    cached_scaler_features = int(getattr(nn_x_scaler, "n_features_in_", -1))
    cached_model_features = int(getattr(nn_forward_model, "n_features_in_", -1))
    expected_features = int(len(param_names))

    artifact_compatible = (
        cached_param_names == list(param_names)
        and cached_soh_cols == list(soh_cols)
        and (not cached_target_cols or len(cached_target_cols) == len(soh_cols))
        and cached_target_encoding == "linear_cycle_monotonic_delta"
        and cached_scaler_features == expected_features
        and cached_model_features == expected_features
    )

    if artifact_compatible:
        print(f"Loaded cached emulator artifact from {ARTIFACT_PATH}")
    else:
        print("Cached artifact is incompatible with the current SOH-only notebook configuration.")
        print(f"Expected {expected_features} input features; cached scaler/model have {cached_scaler_features}/{cached_model_features}.")
        print("Retraining a fresh artifact with the current target layout...")
        force_retrain = True

if (not ARTIFACT_PATH.exists()) or force_retrain:
    nn_forward_model = MLPRegressor(
        hidden_layer_sizes=(64, 64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=5000,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=100,
        random_state=RANDOM_STATE,
    )
    nn_forward_model.fit(X_train_nn, Y_train_nn)
    artifact_meta = {
        "model_tag": MODEL_TAG,
        "random_state": RANDOM_STATE,
        "param_names": param_names,
        "soh_cols": soh_cols,
        "extra_target_cols": extra_target_cols,
        "target_cols": nn_target_cols,
        "n_train": int(len(X_train_raw)),
        "n_test": int(len(X_test_raw)),
        "dataset_root": str(PBS_DATA_ROOT),
        "parameter_representation": PARAMETER_REPRESENTATION,
        "soh_thresholds": list(dataset.soh_thresholds),
        "target_encoding": "linear_cycle_monotonic_delta",
    }
    artifact = {
        "model": nn_forward_model,
        "x_scaler": nn_x_scaler,
        "y_scaler": nn_y_scaler,
        "metadata": artifact_meta,
    }
    joblib.dump(artifact, ARTIFACT_PATH)
    print(f"Trained and saved emulator artifact to {ARTIFACT_PATH}")


def nn_forward_predict_monotonic_from_normalized(X_norm):
    y_pred_scaled = nn_forward_model.predict(X_norm)
    return nn_y_scaler.inverse_transform(y_pred_scaled)


def nn_forward_predict_raw_from_normalized(X_norm):
    y_pred_monotonic = nn_forward_predict_monotonic_from_normalized(X_norm)
    return invert_monotonic_forward_targets(y_pred_monotonic, n_soh_targets)


def nn_forward_predict_raw_from_raw(X_raw):
    X_norm = nn_x_scaler.transform(np.asarray(X_raw, dtype=float))
    return nn_forward_predict_raw_from_normalized(X_norm)


## Module 8. Forward-emulator accuracy summary


In [ ]:
Y_pred_raw = nn_forward_predict_raw_from_normalized(X_test_nn)
monotonic_ok_mask = monotonic_ok_mask_from_raw_predictions(Y_pred_raw, n_soh_targets)

metric_rows = []
for j, col in enumerate(nn_target_cols):
    yt = Y_test_raw[:, j]
    yp = Y_pred_raw[:, j]
    metric_rows.append({
        "target": col,
        "R2": r2_score(yt, yp),
        "RMSE": rmse(yt, yp),
        "MAE": mean_absolute_error(yt, yp),
    })
metrics_df = pd.DataFrame(metric_rows)
#metrics_df.to_csv(METRICS_CSV_PATH, index=False)

overall_soh_metrics = pd.DataFrame([{
    "scope": "all_soh_targets",
    "R2": r2_score(Y_test_raw.ravel(), Y_pred_raw.ravel()),
    "RMSE": rmse(Y_test_raw.ravel(), Y_pred_raw.ravel()),
    "MAE": mean_absolute_error(Y_test_raw.ravel(), Y_pred_raw.ravel()),
}])

print(f"Iterations: {getattr(nn_forward_model, 'n_iter_', 'NA')}")
print(f"Monotonic test predictions: {int(monotonic_ok_mask.sum())}/{len(monotonic_ok_mask)} = {monotonic_ok_mask.mean():.1%}")
display(overall_soh_metrics)
display(metrics_df)


## Module 9. Visualization: all SOH true vs predicted


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
from matplotlib.colors import Normalize

# ---------- Data preparation ----------
y_true_cycle = Y_test_raw[:, :n_soh_targets].ravel()
y_pred_cycle = Y_pred_raw[:, :n_soh_targets].ravel()

max_cycle = max(y_true_cycle.max(), y_pred_cycle.max())
y_true_norm = y_true_cycle / max_cycle
y_pred_norm = y_pred_cycle / max_cycle

rmse = np.sqrt(mean_squared_error(y_true_norm, y_pred_norm))
r2 = r2_score(y_true_norm, y_pred_norm)

# Compute errors (in units of 1e-3)
errors = (y_pred_norm - y_true_norm) * 1000

# Normalize errors to [0, 1] for colour mapping
norm_errors = (errors - errors.min()) / (errors.max() - errors.min())

lo, hi = 0.0, 1.0
margin = 0.05

# ---------- Create a dark blue colour map (light blue for small residuals, dark blue for large) ----------
# Use 'Blues' but slice it to start from a medium-light blue (0.4) to dark blue (1.0)
blue_cmap = plt.cm.Blues_r
# Create a new colormap by taking a subset of the original (from 0.4 to 1.0)
blue_cmap_subset = blue_cmap.from_list(
    'dark_blues_subset',
    [blue_cmap(0.0), blue_cmap(0.4)],
    N=256
)

# Normalize errors for the scatter plot (so the colour mapping uses the full range of errors)
norm_scatter = Normalize(vmin=errors.min(), vmax=errors.max())

fig, ax = plt.subplots(figsize=(5, 4), dpi=500)

# ---------- Scatter plot with colour mapped to error ----------
sc = ax.scatter(y_true_norm, y_pred_norm, s=40, alpha=0.8,
                edgecolor="None", c=errors, cmap=blue_cmap_subset, norm=norm_scatter)

ax.plot([lo, hi], [lo, hi], color="black", linestyle="--", linewidth=1.1)

ax.set_xlabel("True cycle (normalized)", fontsize=12)
ax.set_ylabel("Emulator predicted cycle (normalized)", fontsize=12)
ax.set_xlim(lo - margin, hi + margin)
ax.set_ylim(lo - margin, hi + margin)
ax.set_aspect("equal", adjustable="box")

# ---------- Text box in the bottom-right corner ----------
ax.text(0.95, 0.05, f'RMSE = {rmse:.4f}\n $R^2$ = {r2:.4f}',
        transform=ax.transAxes, ha='right', va='bottom', linespacing=1.5,
        fontsize=12)

# ---------- Inset subplot in the top-left corner ----------
left, bottom = 0.15, 0.75
width, height = 0.30, 0.20
ax_inset = ax.inset_axes([left, bottom, width, height])

# Compute histogram counts and bin edges
counts, bin_edges = np.histogram(errors, bins=300)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]

# Normalise counts for colour mapping (0 to 1)
norm_counts = Normalize(vmin=counts.min(), vmax=counts.max())
counts_norm = norm_counts(counts)

# Plot each bar individually with colour mapped to its count
for center, count, c_norm in zip(bin_centers, counts, counts_norm):
    color = blue_cmap_subset(c_norm)  # map normalised count to colour
    ax_inset.bar(center, count, width=bin_width, color=color, edgecolor='white', linewidth=0.3)

# Remove top and right spines
ax_inset.spines['top'].set_visible(False)
ax_inset.spines['right'].set_visible(False)
ax_inset.spines['left'].set_linewidth(0.8)
ax_inset.spines['bottom'].set_linewidth(0.8)

ax_inset.set_xlabel('Residual ($10^{-3}$)', fontsize=8)
ax_inset.set_xlim(-12, 12)
ax_inset.set_ylim(0, 320)
ax_inset.set_xticks([-10, -5, 0, 5, 10])
ax_inset.set_yticks([0, 150, 300])
ax_inset.set_xticklabels([-10, -5, 0, 5, 10], fontsize=8)
ax_inset.set_yticklabels([0, 150, 300], fontsize=8)

# No colorbar added (as requested)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "F2b.tiff", dpi=500, format="tiff", bbox_inches="tight")
plt.show()

print(f"original scale RMSE (cycle): {np.sqrt(mean_squared_error(y_true_cycle, y_pred_cycle)):.2f} (normalized): {rmse:.4f}")
